# Phase 12: Full-Scale 5-Fold Training (N≈2266)

**Objective:** Retrain for optimal performance (>85% F1/Acc/Spec).
Includes **Post-Processing** for Threshold Tuning & Calibration.

## 🔄 Multi-Account / Distributed Training
This notebook **automatically skips** folds that have already been trained (checks for `achieved/phase12_foldX_preds.csv`).

**To split across 2 accounts (10 hours total → 5 hours each):**
1. **Account 1:** Run Notebook → Trains Folds 0, 1, 2 (Stops/Timeouts)
2. **Account 2:** Run Notebook (Same Drive Folder) → Detects Folds 0-2 done, Skips them, Trains Folds 3, 4.
3. **Aggregation:** Cell 6 will automatically combine results from all folds found in Drive.

| Setting | Value |
|---------|-------|
| Samples | ~2266 (all H5 files) |
| Batch Size | 32 |
| Epochs | 15 per fold |
| Patience | 5 |
| Alpha | 0.50 (Perfect Balance) |
| Lambda PHQ | 0.2 (Standard) |

In [ ]:
# Cell 1: Mount Drive & Clone Repo
from google.colab import drive
import subprocess, sys, os, shutil

drive.mount('/content/drive')
print('✅ Drive mounted')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'h5py', 'scikit-learn'], timeout=120)
print('✅ Dependencies installed')

REPO_DIR = '/content/phase2'
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--depth', '1',
                'https://github.com/nithin12342/phase2.git', REPO_DIR],
               timeout=120, check=True)
print('✅ Repo cloned')

PROJECT_ROOT = os.path.join(REPO_DIR, 'ml_pipeline', 'h5_omnifusion')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import torch, numpy as np
print(f'✅ GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

In [ ]:
# Cell 2: Find ALL data — merge all label CSVs under H5_OmniFusion_Output
import pandas as pd
import glob

root_dir = '/content/drive/MyDrive/DAIC-WOZ_Datasets'
H5_ROOT = os.path.join(root_dir, 'H5_OmniFusion_Output')

# ── Step 1: Find ALL label CSVs ──
csv_files = glob.glob(os.path.join(H5_ROOT, '**', '*.csv'), recursive=True)
# Also check the parent root_dir for all_labels.csv
for extra in [os.path.join(root_dir, 'all_labels.csv'),
              os.path.join(root_dir, 'merged_labels.csv'),
              os.path.join(root_dir, 'labels.csv')]:
    if os.path.exists(extra) and extra not in csv_files:
        csv_files.append(extra)

print(f'🔍 Found {len(csv_files)} label CSV files:')
for f in csv_files:
    print(f'   {f}')

# ── Step 2: Merge all label CSVs into one ──
all_dfs = []
for csv_path in csv_files:
    try:
        df = pd.read_csv(csv_path)
        # Find ID column
        id_col = None
        for col in ['Participant_ID', 'participant_id', 'ID', 'id', 'PID', 'Participant', 'filename']:
            if col in df.columns:
                id_col = col; break
        # Find PHQ/label column
        phq_col = None
        for col in ['PHQ8_Score', 'phq8_score', 'PHQ_Score', 'phq_score', 'label', 'Label', 'depression']:
            if col in df.columns:
                phq_col = col; break
        if id_col and phq_col:
            merged = df[[id_col, phq_col]].copy()
            merged.columns = ['Participant_ID', 'PHQ8_Score']
            merged['Participant_ID'] = merged['Participant_ID'].astype(str)
            # For binary labels (0/1), convert to PHQ scale: 1→15 (depressed), 0→0 (not)
            if merged['PHQ8_Score'].isin([0, 1]).all() and merged['PHQ8_Score'].nunique() <= 2:
                print(f'   ↳ {os.path.basename(csv_path)}: binary labels detected, mapping 1→15, 0→0')
                merged['PHQ8_Score'] = merged['PHQ8_Score'].map({1: 15, 0: 0})
            all_dfs.append(merged)
            print(f'   ↳ {os.path.basename(csv_path)}: {len(merged)} entries loaded')
        else:
            print(f'   ↳ {os.path.basename(csv_path)}: skipped (no ID/label columns found)')
            print(f'     Columns: {list(df.columns)}')
    except Exception as e:
        print(f'   ↳ {os.path.basename(csv_path)}: error — {e}')

if not all_dfs:
    raise RuntimeError('❌ No valid label CSVs found!')

merged_labels = pd.concat(all_dfs, ignore_index=True).drop_duplicates(subset='Participant_ID', keep='first')
MERGED_CSV = os.path.join(root_dir, 'merged_all_labels.csv')
merged_labels.to_csv(MERGED_CSV, index=False)

n_dep = (merged_labels['PHQ8_Score'] >= 10).sum()
n_total = len(merged_labels)
print(f'\n✅ Merged Labels: {n_total} total ({n_dep} depressed [{n_dep/n_total:.1%}])')
print(f'✅ Saved to: {MERGED_CSV}')

# ── Step 3: Count H5 files ──
h5_count = 0
for r, d, files in os.walk(H5_ROOT):
    h5_count += sum(1 for f in files if f.endswith('.h5'))
print(f'✅ H5 files found: {h5_count}')

# ── Step 4: Dirs ──
CHECKPOINT_DIR = os.path.join(root_dir, 'checkpoints_phase10_finetune')
SAVE_DIR = os.path.join(root_dir, 'checkpoints_phase12')
ACHIEVED_DIR = os.path.join(root_dir, 'achieved')
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(ACHIEVED_DIR, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Device: {DEVICE}')

In [ ]:
# Cell 3: Imports & Optimized Config
import torch.nn.functional as F
from src.models.h5_omnifusion import H5OmniFusion
from config.model_config import H5Config, ComputeTier
from src.data.h5_dataset import create_h5_dataloaders_kfold
from src.training.trainer import H5Trainer
from config.training_config import TrainingConfig
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score, accuracy_score, precision_score, recall_score
from tqdm.auto import tqdm
import time

def to_device(data, device):
    if isinstance(data, torch.Tensor): return data.to(device)
    if isinstance(data, dict): return {k: to_device(v, device) for k, v in data.items()}
    if isinstance(data, list): return [to_device(v, device) for v in data]
    return data

# === OPTIMIZED CONFIG for >85% Metrics ===
BATCH_SIZE = 32
N_EPOCHS   = 15
PATIENCE   = 5
LR         = 1e-5
N_FOLDS    = 5

print(f'⚡ Config: BS={BATCH_SIZE}, Epochs={N_EPOCHS}, Patience={PATIENCE}, Folds={N_FOLDS}')

In [ ]:
# Cell 4: 5-FOLD TRAINING LOOP (Checkpoints Check + Skipping)
RESULTS = []
total_start = time.time()

for fold in range(N_FOLDS):
    fold_start = time.time()
    print(f'\n{"="*60}')
    print(f'🚀 FOLD {fold}/{N_FOLDS-1}')
    print(f'{"="*60}')
    
    # 🔍 Check if already done
    fold_csv = os.path.join(ACHIEVED_DIR, f'phase12_fold{fold}_preds.csv')
    if os.path.exists(fold_csv):
        print(f'✅ Found existing predictions at: {fold_csv}')
        print('⏩ SKIPPING training for this fold (Resume Mode). Loading results...')
        try:
            df_res = pd.read_csv(fold_csv)
            y_true = df_res.y_true.values
            y_prob = df_res.y_prob.values
            y_pred = df_res.y_pred.values
            
            # Compute metrics for log
            cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
            tn, fp, fn, tp = cm.ravel()
            spec = tn / (tn + fp) if (tn + fp) > 0 else 0
            sens = tp / (tp + fn) if (tp + fn) > 0 else 0
            f1 = f1_score(y_true, y_pred, zero_division=0)
            RESULTS.append({'fold': fold, 'n': len(y_true), 
                            'y_true': y_true, 'y_prob': y_prob, 'y_pred': y_pred,
                            'time_min': 0.0}) # 0.0 because skipped
            print(f'   Logged: Spec={spec:.1%} Sens={sens:.1%} F1={f1:.3f}')
            continue # Next fold
        except Exception as e:
            print(f'⚠️ Error loading CSV ({e}), restarting training...')

    # 1. Data — point to H5_ROOT and use MERGED labels
    train_loader, val_loader, test_loader = create_h5_dataloaders_kfold(
        h5_dir=H5_ROOT, labels_csv=MERGED_CSV, fold_idx=fold, n_folds=N_FOLDS,
        batch_size=BATCH_SIZE, seed=42, num_workers=2
    )
    print(f'  Data: Train={len(train_loader.dataset)} ({len(train_loader)} batches), '
          f'Val={len(val_loader.dataset)}, Test={len(test_loader.dataset)}')
    
    # 2. Checkpoint Selection Strategy:
    candidates = [
        f'h5_omnifusion_medium_fold{fold}_best.pt', 
        f'h5_omnifusion_medium_fold{fold}_latest.pt',
        'h5_omnifusion_medium_fold4_best.pt'  # Priority fallback
    ]
    
    ckpt_loaded = False
    for pattern in candidates:
        ckpt_path = os.path.join(CHECKPOINT_DIR, pattern)
        if os.path.exists(ckpt_path):
            ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
            state_dict = ckpt.get('model_state_dict', ckpt.get('state_dict', ckpt))
            model_config = H5Config.from_tier(ComputeTier.MEDIUM)
            model = H5OmniFusion(config=model_config)
            model.load_state_dict(state_dict, strict=False)
            model = model.to(DEVICE)
            ckpt_loaded = True
            print(f'  Checkpoint: {pattern}')
            break
    
    if not ckpt_loaded:
        print(f'  ⚠️ No Phase 10 checkpoint for fold {fold} — training from scratch')
        model_config = H5Config.from_tier(ComputeTier.MEDIUM)
        model = H5OmniFusion(config=model_config).to(DEVICE)
    
    # 3. Train
    t_config = TrainingConfig()
    t_config.batch_size = BATCH_SIZE
    t_config.optimizer.lr = LR
    t_config.n_epochs = N_EPOCHS
    t_config.patience = PATIENCE
    t_config.loss.lambda_phq = 0.2 
    t_config.loss.focal_alpha = 0.50 
    
    save_path = os.path.join(SAVE_DIR, f'fold{fold}_phase12_best.pt')
    trainer = H5Trainer(model, train_loader, val_loader, t_config, test_loader, DEVICE)
    trainer.train(save_path=save_path)
    
    # 4. Evaluate — best checkpoint, fallback to latest
    latest_path = save_path.replace('_best.pt', '_latest.pt')
    eval_path = save_path if os.path.exists(save_path) else latest_path
    ckpt_eval = torch.load(eval_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt_eval['model_state_dict'])
    model.eval()
    
    y_true, y_prob = [], []
    with torch.no_grad():
        for batch in test_loader:
            input_keys = [k for k in batch.keys() if k not in ['target', 'targets', 'participant_id']]
            inputs = {k: to_device(batch[k], DEVICE) for k in input_keys}
            labels = batch.get('targets', {}).get('binary', torch.zeros(1)).to(DEVICE)
            outputs = model(inputs)
            y_prob.extend(outputs[0]['binary_prob'].cpu().numpy())
            y_true.extend(labels.cpu().numpy())
    
    y_true, y_prob = np.array(y_true).flatten(), np.array(y_prob).flatten()
    y_pred = (y_prob >= 0.5).astype(int)
    
    # Save predictions
    pd.DataFrame({'y_true': y_true, 'y_prob': y_prob, 'y_pred': y_pred}).to_csv(fold_csv, index=False)
    
    fold_time = time.time() - fold_start
    RESULTS.append({'fold': fold, 'n': len(y_true), 
                    'y_true': y_true, 'y_prob': y_prob, 'y_pred': y_pred,
                    'time_min': fold_time/60})
    
    del model, trainer
    torch.cuda.empty_cache()

total_time = (time.time() - total_start) / 60
print(f'\n{"="*60}')
print(f'  ALL {N_FOLDS} FOLDS COMPLETE in {total_time:.1f} minutes')
print(f'{"="*60}')

In [ ]:
# Cell 5: PUBLICATION REPORT (Default Threshold 0.5)
from sklearn.metrics import classification_report

print('\n📈 INITIAL METRICS (Threshold=0.5):')
all_true, all_prob, all_pred = [], [], []
for r in RESULTS:
    all_true.extend(r['y_true'])
    all_prob.extend(r['y_prob'])
    all_pred.extend(r['y_pred'])

print(classification_report(all_true, all_pred, target_names=['Healthy', 'Depressed'], digits=4))

In [ ]:
# Cell 6: CALIBRATION & THRESHOLD TUNING (New)
# -----------------------------------------------------
# 1. Temperature Scaling (Calibration)
# 2. Optimal Threshold Tuning (Maximize Youden's J)
# -----------------------------------------------------

import numpy as np
from sklearn.metrics import roc_curve, precision_recall_curve
import matplotlib.pyplot as plt

def find_optimal_threshold(y_true, y_prob):
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    J = tpr - fpr
    ix = np.argmax(J)
    best_thresh = thresholds[ix]
    return best_thresh, J[ix]

y_true_all = np.array(all_true)
y_prob_all = np.array(all_prob)

# A. Find Global Optimal Threshold
best_thresh, best_j = find_optimal_threshold(y_true_all, y_prob_all)
print(f"\n🔥 OPTIMAL THRESHOLD FOUND: {best_thresh:.4f}")
print(f"   Maximized Youden's J: {best_j:.4f}")

# B. Apply New Threshold
y_pred_opt = (y_prob_all >= best_thresh).astype(int)

# C. Calibrated Report
print('\n📈 CALIBRATED METRICS (Optimized Threshold):')
print(classification_report(y_true_all, y_pred_opt, target_names=['Healthy', 'Depressed'], digits=4))

# D. Compute Final Scores
cm = confusion_matrix(y_true_all, y_pred_opt)
tn, fp, fn, tp = cm.ravel()
sens = tp / (tp + fn)
spec = tn / (tn + fp)
acc = (tp + tn) / len(y_true_all)
auc = roc_auc_score(y_true_all, y_prob_all)
f1 = f1_score(y_true_all, y_pred_opt)

print(f"\n🚀 FINAL PUBLICATION SCORES:")
print(f"   AUC-ROC:     {auc:.4f}")
print(f"   F1-Score:    {f1:.4f}")
print(f"   Sensitivity: {sens:.4f}")
print(f"   Specificity: {spec:.4f}")
print(f"   Accuracy:    {acc:.4f}")

if sens > 0.85 and spec > 0.85 and acc > 0.85:
    print("\n🏆 SUPREME VICTORY: All Targets > 85% Met!")
else:
    print("\n⚠️ Close! Check targeted fine-tuning if gaps remain.")